In [ ]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 3.3 MB/s eta 0:00:00


In [ ]:
# qwen_daily_batch_classifier
import os
import json
import time
import re
import pandas as pd
from groq import Groq
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report

# ========== CONFIGURATION ==========
API_KEY = "******************"
DATASET_FILE = "/content/drive/MyDrive/Colab Notebooks/dataset/task2_test_gold_label_final.json"
FEW_SHOT_PROMPT_FILE = "/content/drive/MyDrive/Colab Notebooks/dataset/few_shot_prompt_cleaned.txt"
RESULT_DIR = "/content/drive/MyDrive/Colab Notebooks/qwen_batch_results_short_new"
DAILY_LIMIT = 70
TIME_BETWEEN_REQUESTS = 10

os.makedirs(RESULT_DIR, exist_ok=True)

# ========== SETUP GROQ ==========
client = Groq(api_key=API_KEY)
MODEL_NAME = "qwen/qwen3-32b"

# ========== STEP 1: LOAD FULL DATA ==========
try:
    with open(DATASET_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"✅ Dataset type: {type(data)} | Entries: {len(data)}")
except Exception as e:
    print("❌ ERROR: Failed loading dataset file.")
    raise e

# ========== STEP 2: LOAD FEW-SHOT PROMPT ==========
try:
    with open(FEW_SHOT_PROMPT_FILE, "r", encoding="utf-8") as f:
        FEW_SHOT_LIST = f.read()
    print(f"✅ Loaded few-shot examples ({len(FEW_SHOT_LIST)} characters)")
except Exception as e:
    print("❌ ERROR: Failed loading few-shot file.")
    raise e

# ========== STEP 3: TEXT CLEANING ==========
def clean_text(text):
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"@\S+", " ", text)
    text = re.sub(r"&\w+;", " ", text)
    #text = re.sub(r"#", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ========== STEP 4: FIND UNPROCESSED BATCH ==========
processed_ids = set()
try:
    for fname in os.listdir(RESULT_DIR):
        if fname.endswith(".json"):
            path = os.path.join(RESULT_DIR, fname)
            try:
                with open(path, encoding="utf-8") as f:
                    processed = json.load(f)
                    processed_ids.update([entry["id"] for entry in processed])
            except json.JSONDecodeError as je:
                print(f"⚠️ Skipping invalid result file: {fname} ({je})")
except Exception as e:
    print("❌ ERROR while reading processed files.")
    raise e

remaining_data = [entry for entry in data if entry["id"] not in processed_ids]
if not remaining_data:
    print("✅ All entries processed.")
    exit()

batch = remaining_data[:DAILY_LIMIT]
batch_df = pd.DataFrame(batch)
batch_df["actual_techniques"] = batch_df["labels"].apply(
    lambda lst: list(set(l["technique"] for l in lst))
)

# ========== STEP 5: CLASSIFICATION FUNCTION ==========
def classify_with_few_shot(text):
    prompt = f"""
You are a classifier for propaganda techniques in Arabic tweets.

Here are a few examples of tweets with their correct classifications:
{FEW_SHOT_LIST}

Now analyze the following new tweet:
{text}

Instructions:
- Identify the propaganda techniques used in this tweet.
- Return only the list of detected techniques.
- The output must be valid JSON.
- If no techniques are detected, return an empty list: []
- Example format: ["Appeal to Authority", "Name Calling/Labeling"]

Output:
"""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        content = response.choices[0].message.content.strip()

        #  1. Remove any <think>...</think> blocks
        content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()

        #  2. Find the last JSON-like list in the output
        matches = re.findall(r"\[.*?\]", content, re.DOTALL)
        if matches:
            last_json = matches[-1]
            try:
                parsed = json.loads(last_json)
                return parsed
            except json.JSONDecodeError:
                print(f"⚠️ JSONDecodeError (after cleaning). Raw content:\n{content}")
                return []
        else:
            print(f"⚠️ No JSON list found. Raw content:\n{content}")
            return []

    except Exception as e:
        print(f"❌ Error in classify_with_few_shot: {e}")
        return []


# ========== STEP 6: RUN CLASSIFICATION ==========
predictions = []
start_time = time.time()

try:
    for idx, row in batch_df.iterrows():
        print(f"\n🟩 Processing tweet ID: {row['id']} ...")
        pred = classify_with_few_shot(row["text"])
        predictions.append(pred)
        print(f"✅ Prediction for tweet ID {row['id']}: {pred}")
        time.sleep(TIME_BETWEEN_REQUESTS)
except KeyboardInterrupt:
    print("⛔ Interrupted manually.")
except Exception as e:
    print("❌ ERROR during classification loop:")
    raise e

end_time = time.time()
batch_df["predicted_techniques"] = predictions

# ========== STEP 7: SAVE RESULTS ==========
batch_id = len(os.listdir(RESULT_DIR)) + 1
output_path = os.path.join(RESULT_DIR, f"results_batch_{batch_id}.json")
try:
    tmp_path = output_path + ".tmp"
    batch_df[["id", "text", "actual_techniques", "predicted_techniques"]].to_json(
        tmp_path, force_ascii=False, indent=2, orient="records"
    )
    os.replace(tmp_path, output_path)
    print(f"✅ Batch {batch_id} saved successfully → {output_path}")
except Exception as e:
    print("❌ ERROR: Failed to save results.")
    raise e

# ========== STEP 8: EVALUATION ==========
try:
    mlb = MultiLabelBinarizer()
    y_true = mlb.fit_transform(batch_df["actual_techniques"])
    y_pred = mlb.transform(batch_df["predicted_techniques"])
    report = classification_report(y_true, y_pred, target_names=mlb.classes_)
    print("\n\033[1mTechnique Classification Report (Daily Batch):\033[0m")
    print(report)
except Exception as e:
    print("❌ ERROR in evaluation step.")
    raise e

print(f"🕒 Time taken: {end_time - start_time:.2f} seconds")
print("✅ All steps completed successfully.")


✅ Dataset type: <class 'list'> | Entries: 322
✅ Loaded few-shot examples (1821 characters)

🟩 Processing tweet ID: 1296405056682590208 ...
✅ Prediction for tweet ID 1296405056682590208: ['Loaded Language']

🟩 Processing tweet ID: 1396831800660480007 ...
✅ Prediction for tweet ID 1396831800660480007: []

🟩 Processing tweet ID: 1391808358936612866 ...
✅ Prediction for tweet ID 1391808358936612866: ['Appeal to fear/prejudice', 'Loaded Language']

🟩 Processing tweet ID: 1395344678120214528 ...
✅ Prediction for tweet ID 1395344678120214528: []

🟩 Processing tweet ID: 1386024412869275650 ...
✅ Prediction for tweet ID 1386024412869275650: []

🟩 Processing tweet ID: 1399162717135573002 ...
✅ Prediction for tweet ID 1399162717135573002: []

🟩 Processing tweet ID: 1374443328998113280 ...
✅ Prediction for tweet ID 1374443328998113280: ['Loaded Language']

🟩 Processing tweet ID: 1328008800230658048 ...
✅ Prediction for tweet ID 1328008800230658048: []

🟩 Processing tweet ID: 1370413777695432714 ..

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [ ]:
# llama-3.3-70b-versatile_daily_batch_classifier
import os
import json
import time
import re
import pandas as pd
from groq import Groq
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report

# ========== CONFIGURATION ==========
API_KEY = "*********************"

DATASET_FILE = "/content/drive/MyDrive/Colab Notebooks/dataset/task2_test_gold_label_final.json"
FEW_SHOT_PROMPT_FILE = "/content/drive/MyDrive/Colab Notebooks/dataset/few_shot_prompt_cleaned.txt"
RESULT_DIR = "/content/drive/MyDrive/Colab Notebooks/llama_batch_results_short_new"
DAILY_LIMIT = 50
TIME_BETWEEN_REQUESTS = 10

os.makedirs(RESULT_DIR, exist_ok=True)

# ========== SETUP GROQ ==========
client = Groq(api_key=API_KEY)
MODEL_NAME = "llama-3.3-70b-versatile"

# ========== STEP 1: LOAD FULL DATA ==========
try:
    with open(DATASET_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"✅ Dataset type: {type(data)} | Entries: {len(data)}")
except Exception as e:
    print("❌ ERROR: Failed loading dataset file.")
    raise e

# ========== STEP 2: LOAD FEW-SHOT PROMPT ==========
try:
    with open(FEW_SHOT_PROMPT_FILE, "r", encoding="utf-8") as f:
        FEW_SHOT_LIST = f.read()
    print(f"✅ Loaded few-shot examples ({len(FEW_SHOT_LIST)} characters)")
except Exception as e:
    print("❌ ERROR: Failed loading few-shot file.")
    raise e

# ========== STEP 3: TEXT CLEANING ==========
def clean_text(text):
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"@\S+", " ", text)
    text = re.sub(r"&\w+;", " ", text)
    #text = re.sub(r"#", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ========== STEP 4: FIND UNPROCESSED BATCH ==========
processed_ids = set()
try:
    for fname in os.listdir(RESULT_DIR):
        if fname.endswith(".json"):
            path = os.path.join(RESULT_DIR, fname)
            try:
                with open(path, encoding="utf-8") as f:
                    processed = json.load(f)
                    processed_ids.update([entry["id"] for entry in processed])
            except json.JSONDecodeError as je:
                print(f"⚠️ Skipping invalid result file: {fname} ({je})")
except Exception as e:
    print("❌ ERROR while reading processed files.")
    raise e

remaining_data = [entry for entry in data if entry["id"] not in processed_ids]
if not remaining_data:
    print("✅ All entries processed.")
    exit()

batch = remaining_data[:DAILY_LIMIT]
batch_df = pd.DataFrame(batch)
batch_df["actual_techniques"] = batch_df["labels"].apply(
    lambda lst: list(set(l["technique"] for l in lst))
)

# ========== STEP 5: CLASSIFICATION FUNCTION ==========
def classify_with_few_shot(text):
    prompt = f"""
You are a classifier for propaganda techniques in Arabic tweets.

Here are a few examples of tweets with their correct classifications:
{FEW_SHOT_LIST}

Now analyze the following new tweet:
{text}

Instructions:
- Identify the propaganda techniques used in this tweet.
- Return only the list of detected techniques.
- The output must be valid JSON.
- If no techniques are detected, return an empty list: []
- Example format: ["Appeal to Authority", "Name Calling/Labeling"]

Output:
"""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        content = response.choices[0].message.content.strip()

        # 🧹 1. Remove any <think>...</think> blocks
        content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()

        # 🧩 2. Find the last JSON-like list in the output
        matches = re.findall(r"\[.*?\]", content, re.DOTALL)
        if matches:
            last_json = matches[-1]
            try:
                parsed = json.loads(last_json)
                return parsed
            except json.JSONDecodeError:
                print(f"⚠️ JSONDecodeError (after cleaning). Raw content:\n{content}")
                return []
        else:
            print(f"⚠️ No JSON list found. Raw content:\n{content}")
            return []

    except Exception as e:
        print(f"❌ Error in classify_with_few_shot: {e}")
        return []


# ========== STEP 6: RUN CLASSIFICATION ==========
predictions = []
start_time = time.time()

try:
    for idx, row in batch_df.iterrows():
        print(f"\n🟩 Processing tweet ID: {row['id']} ...")
        pred = classify_with_few_shot(row["text"])
        predictions.append(pred)
        print(f"✅ Prediction for tweet ID {row['id']}: {pred}")
        time.sleep(TIME_BETWEEN_REQUESTS)
except KeyboardInterrupt:
    print("⛔ Interrupted manually.")
except Exception as e:
    print("❌ ERROR during classification loop:")
    raise e

end_time = time.time()
batch_df["predicted_techniques"] = predictions

# ========== STEP 7: SAVE RESULTS ==========
batch_id = len(os.listdir(RESULT_DIR)) + 1
output_path = os.path.join(RESULT_DIR, f"results_batch_{batch_id}.json")
try:
    tmp_path = output_path + ".tmp"
    batch_df[["id", "text", "actual_techniques", "predicted_techniques"]].to_json(
        tmp_path, force_ascii=False, indent=2, orient="records"
    )
    os.replace(tmp_path, output_path)
    print(f"✅ Batch {batch_id} saved successfully → {output_path}")
except Exception as e:
    print("❌ ERROR: Failed to save results.")
    raise e


try:
    mlb = MultiLabelBinarizer()
    y_true = mlb.fit_transform(batch_df["actual_techniques"])
    y_pred = mlb.transform(batch_df["predicted_techniques"])
    report = classification_report(y_true, y_pred, target_names=mlb.classes_)
    print("\n\033[1mTechnique Classification Report (Daily Batch):\033[0m")
    print(report)
except Exception as e:
    print("❌ ERROR in evaluation step.")
    raise e

print(f"🕒 Time taken: {end_time - start_time:.2f} seconds")
print("✅ All steps completed successfully.")


✅ Dataset type: <class 'list'> | Entries: 322
✅ Loaded few-shot examples (1821 characters)

🟩 Processing tweet ID: 1390879132783939584 ...
✅ Prediction for tweet ID 1390879132783939584: []

🟩 Processing tweet ID: 1371494519326310401 ...
✅ Prediction for tweet ID 1371494519326310401: ['Loaded Language', 'Smears']

🟩 Processing tweet ID: 1381578099066859532 ...
✅ Prediction for tweet ID 1381578099066859532: ['Appeal to Authority']

🟩 Processing tweet ID: 1393857504099045376 ...
✅ Prediction for tweet ID 1393857504099045376: ['Loaded Language']

🟩 Processing tweet ID: 1382257854984376324 ...
✅ Prediction for tweet ID 1382257854984376324: ['Appeal to Authority']

🟩 Processing tweet ID: 1388222625990774788 ...
✅ Prediction for tweet ID 1388222625990774788: []

🟩 Processing tweet ID: 1397470041512235013 ...
✅ Prediction for tweet ID 1397470041512235013: ['Loaded Language']

🟩 Processing tweet ID: 1349634468806619136 ...
✅ Prediction for tweet ID 1349634468806619136: []

🟩 Processing tweet ID

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['Appeal to Authority'] will be ignored
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` p

In [ ]:
import os
import json
import pandas as pd

# Path to all batch result files
RESULTS_DIR = "/content/drive/MyDrive/Colab Notebooks/llama_batch_results_short_new_0.7"
MERGED_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset/llama_batch_results_short_merged_new.json"

# Collect all JSON files
all_data = []
for file in sorted(os.listdir(RESULTS_DIR)):
    if file.endswith(".json"):
        with open(os.path.join(RESULTS_DIR, file), "r", encoding="utf-8") as f:
            data = json.load(f)
            all_data.extend(data)

# Save the merged file
with open(MERGED_PATH, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"✅ Merged {len(all_data)} entries into: {MERGED_PATH}")


✅ Merged 322 entries into: /content/drive/MyDrive/Colab Notebooks/dataset/llama_batch_results_short_merged_new.json


In [ ]:
import json

# Load gold file (contains IDs)
with open("/content/drive/MyDrive/Colab Notebooks/dataset/task2_test_gold_label_final.json", "r", encoding="utf-8") as f:
    gold_data = json.load(f)

# Load predictions file (missing IDs)
with open("/content/drive/MyDrive/Colab Notebooks/dataset/deepseek.json", "r", encoding="utf-8") as f:
    pred_data = json.load(f)

# Ensure both lists are same length
if len(gold_data) != len(pred_data):
    raise ValueError("gold.json and pred.json do NOT have the same number of entries.")

# Add IDs to prediction entries
updated_pred = []
for gold_item, pred_item in zip(gold_data, pred_data):
    pred_item["id"] = gold_item["id"]
    updated_pred.append(pred_item)

# Save updated file
with open("/content/drive/MyDrive/Colab Notebooks/dataset/pred_with_ids.json", "w", encoding="utf-8") as f:
    json.dump(updated_pred, f, ensure_ascii=False, indent=4)

print("Finished! New file saved as pred_with_ids.json")


Finished! New file saved as pred_with_ids.json


In [ ]:
#gpt5
import json
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report

# Load actual and predicted JSON files
with open("/content/drive/MyDrive/Colab Notebooks/dataset/task2_test_gold_label_final.json", "r", encoding="utf-8") as f:
    actual_data = json.load(f)

with open("/content/drive/MyDrive/Colab Notebooks/dataset/gpt_5_pred_with_ids_0.0.json", "r", encoding="utf-8") as f:
    predicted_data = json.load(f)

# Create dictionaries for fast access by tweet id
actual_dict = {item["id"]: [label["technique"] for label in item.get("labels", [])] for item in actual_data}
predicted_dict = {item["id"]: item.get("techniques", []) for item in predicted_data}

# Get the union of all tweet ids
all_ids = set(actual_dict.keys()) | set(predicted_dict.keys())

# Prepare lists for multilabel binarization
y_true = []
y_pred = []

for tweet_id in all_ids:
    y_true.append(actual_dict.get(tweet_id, []))  # empty list if not present
    y_pred.append(predicted_dict.get(tweet_id, []))

# Convert labels to binary format
mlb = MultiLabelBinarizer()
y_true_bin = mlb.fit_transform(y_true)
y_pred_bin = mlb.transform(y_pred)  # transform with the same classes

# Generate classification report
report = classification_report(y_true_bin, y_pred_bin, target_names=mlb.classes_, zero_division=0)
print(report)


                                               precision    recall  f1-score   support

                          Appeal to authority       0.00      0.00      0.00         1
                     Appeal to fear/prejudice       0.29      0.71      0.41        24
         Black-and-white Fallacy/Dictatorship       0.00      0.00      0.00         7
                    Causal Oversimplification       0.50      0.25      0.33         4
                                        Doubt       0.50      0.37      0.42        19
                    Exaggeration/Minimisation       0.19      0.22      0.20        23
                                  Flag-waving       0.23      0.78      0.35         9
             Glittering generalities (Virtue)       0.03      1.00      0.05         1
                              Loaded Language       0.82      0.60      0.69       220
                        Name calling/Labeling       0.87      0.19      0.32       134
Obfuscation, Intentional vagueness, Confus

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['Emotional appeal'] will be ignored
  warnings.warn(


In [ ]:
#deepseek
import json
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report

# Load actual and predicted JSON files
with open("/content/drive/MyDrive/Colab Notebooks/dataset/task2_test_gold_label_final.json", "r", encoding="utf-8") as f:
    actual_data = json.load(f)

with open("/content/drive/MyDrive/Colab Notebooks/dataset/deepseek_pred_with_ids.json", "r", encoding="utf-8") as f:
    predicted_data = json.load(f)

# Create dictionaries for fast access by tweet id
actual_dict = {item["id"]: [label["technique"] for label in item.get("labels", [])] for item in actual_data}
predicted_dict = {item["id"]: item.get("techniques", []) for item in predicted_data}

# Get the union of all tweet ids
all_ids = set(actual_dict.keys()) | set(predicted_dict.keys())

# Prepare lists for multilabel binarization
y_true = []
y_pred = []

for tweet_id in all_ids:
    y_true.append(actual_dict.get(tweet_id, []))  # empty list if not present
    y_pred.append(predicted_dict.get(tweet_id, []))

# Convert labels to binary format
mlb = MultiLabelBinarizer()
y_true_bin = mlb.fit_transform(y_true)
y_pred_bin = mlb.transform(y_pred)  # transform with the same classes

# Generate classification report
report = classification_report(y_true_bin, y_pred_bin, target_names=mlb.classes_, zero_division=0)
print(report)


                                               precision    recall  f1-score   support

                          Appeal to authority       0.00      0.00      0.00         1
                     Appeal to fear/prejudice       0.30      0.92      0.45        24
         Black-and-white Fallacy/Dictatorship       1.00      0.29      0.44         7
                    Causal Oversimplification       0.00      0.00      0.00         4
                                        Doubt       0.42      0.42      0.42        19
                    Exaggeration/Minimisation       0.62      0.22      0.32        23
                                  Flag-waving       0.19      0.78      0.30         9
             Glittering generalities (Virtue)       0.14      1.00      0.25         1
                              Loaded Language       0.82      0.66      0.73       220
                        Name calling/Labeling       0.68      0.39      0.49       134
Obfuscation, Intentional vagueness, Confus

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report
import pandas as pd
import json

MERGED_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset/llama_batch_results_short_merged_new_0.2.json"
# Load the merged results
with open(MERGED_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

df = pd.DataFrame(results)

# Normalize predicted techniques
df["predicted_techniques"] = df["predicted_techniques"].apply(
    lambda lst: list(set([t.strip().lower() for t in lst])) if lst else []
)
df["actual_techniques"] = df["actual_techniques"].apply(
    lambda lst: list(set([t.strip().lower() for t in lst])) if lst else []
)

# Binarize and evaluate
mlb = MultiLabelBinarizer()
y_true = mlb.fit_transform(df["actual_techniques"])
y_pred = mlb.transform(df["predicted_techniques"])

report = classification_report(y_true, y_pred, target_names=mlb.classes_)
print("\n\033[1m📊 Final Technique Classification Report:\033[0m")
print(report)



📊 Final Technique Classification Report:
                                               precision    recall  f1-score   support

                          appeal to authority       0.00      0.00      0.00         1
                     appeal to fear/prejudice       0.22      0.83      0.35        24
         black-and-white fallacy/dictatorship       0.00      0.00      0.00         7
                    causal oversimplification       0.00      0.00      0.00         4
                                        doubt       0.00      0.00      0.00        19
                    exaggeration/minimisation       0.20      0.43      0.27        23
                                  flag-waving       0.00      0.00      0.00         9
             glittering generalities (virtue)       0.00      0.00      0.00         1
                              loaded language       0.79      0.51      0.62       220
                        name calling/labeling       0.63      0.45      0.52       134


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['appeal to emotion/prejudice', 'appeal to nationalism/prejudice'] will be ignored
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report
import pandas as pd
import json

MERGED_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset/qwen_batch_results_short_merged_new_0.json"
# Load the merged results
with open(MERGED_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

df = pd.DataFrame(results)

# Normalize predicted techniques
df["predicted_techniques"] = df["predicted_techniques"].apply(
    lambda lst: list(set([t.strip().lower() for t in lst])) if lst else []
)
df["actual_techniques"] = df["actual_techniques"].apply(
    lambda lst: list(set([t.strip().lower() for t in lst])) if lst else []
)

# Binarize and evaluate
mlb = MultiLabelBinarizer()
y_true = mlb.fit_transform(df["actual_techniques"])
y_pred = mlb.transform(df["predicted_techniques"])

report = classification_report(y_true, y_pred, target_names=mlb.classes_)
print("\n\033[1m📊 Final Technique Classification Report:\033[0m")
print(report)



📊 Final Technique Classification Report:
                                               precision    recall  f1-score   support

                          appeal to authority       0.00      0.00      0.00         1
                     appeal to fear/prejudice       0.32      0.62      0.42        24
         black-and-white fallacy/dictatorship       0.00      0.00      0.00         7
                    causal oversimplification       0.00      0.00      0.00         4
                                        doubt       0.00      0.00      0.00        19
                    exaggeration/minimisation       0.19      0.30      0.23        23
                                  flag-waving       0.00      0.00      0.00         9
             glittering generalities (virtue)       0.00      0.00      0.00         1
                              loaded language       0.76      0.56      0.65       220
                        name calling/labeling       0.79      0.40      0.53       134


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['appeal to emotion'] will be ignored
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` par